# OneVoice V2 — physical data audit
Clone source from GitHub and audit `MyDrive/onevoice_audio_v1`. If `manifest.jsonl` is missing, recover deterministic fields from filenames. The full audit opens metadata for 24,192 WAV files on Drive, uses 8 workers and prints progress every 500 files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'soundfile'], check=True)
DATASET_ROOT = MYDRIVE / 'onevoice_audio_v1'
MANIFEST = DATASET_ROOT / 'manifest.jsonl'
REPORT_DIR = WORK_ROOT / 'reports/data_audit_v1'
print('Source:', REPO, '| Data:', DATASET_ROOT, '| Reports:', REPORT_DIR)


## Bước 1 — Phục hồi manifest nếu thiếu
Bước này chỉ lập danh sách tên file một lần. Các lựa chọn ngẫu nhiên của generator V1 như speaker/noise/SNR không thể suy ngược và được đánh dấu `unrecoverable_v1`.

In [ ]:
if not MANIFEST.is_file():
    print('manifest.jsonl is missing; recovering deterministic fields from V1 filenames...')
    subprocess.run([sys.executable, 'scripts/recover_v1_manifest.py', '--dataset-root', str(DATASET_ROOT), '--metadata-csv', 'data/onevoice_construction_v2/utterances_all.csv', '--output', str(MANIFEST)], check=True)
else:
    print('Manifest already exists:', MANIFEST)


## Bước 2 — Audit vật lý toàn bộ WAV
Có thể mất thời gian do Google Drive phải mở 24,192 file nhỏ. Cell sẽ in `checked N/24192`; nếu con số tiếp tục tăng thì không bị treo.

In [ ]:
audit = subprocess.run([sys.executable, 'scripts/audit_audio_dataset.py', str(MANIFEST), '--expected-clean', '8064', '--expected-noisy', '16128', '--workers', '8', '--progress-every', '500', '--report-dir', str(REPORT_DIR)])
print('Audit exit code:', audit.returncode, '(non-zero is expected when V1 speaker identity is unrecoverable)')


In [ ]:
import json
result = json.loads((REPORT_DIR / 'audit.json').read_text(encoding='utf-8'))
display(result)
recovery = DATASET_ROOT / 'manifest_recovery_report.json'
if recovery.is_file(): display(json.loads(recovery.read_text(encoding='utf-8')))
